In [1]:
# ZentroMart AI Customer Support Assistant

## Overview

#This project implements an AI-powered customer support assistant for ZentroMart using Retrieval-Augmented Generation (RAG). The system combines semantic search with Large Language Models to answer customer queries, supports order lookup with verification, and routes dangerous issues to human support.

### Features
# Retrieval-Augmented Generation (RAG)
#Semantic Search using FAISS
# Sentence Transformer Embeddings
# Order Lookup Tool
# Intent Classification
# Human Ticket Escalation

### Tech Stack
# Python
# FAISS
# Sentence Transformers
# LangChain
# OpenRouter
# Kaggle

In [2]:
import os
import numpy as np
import pandas as pd

In [3]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

OPENROUTER_API_KEY = user_secrets.get_secret("OPENROUTER_API_KEY")

print("OpenRouter API key loaded successfully!")

OpenRouter API key loaded successfully!


In [4]:
import os

kb_path = "/kaggle/input/datasets/bolluhansika/capstone-dataset/Capstone_Project_Dataset_FTTT_2026/kb"

documents = []

for file in os.listdir(kb_path):
    if file.endswith(".md"):
        with open(os.path.join(kb_path, file), "r", encoding="utf-8") as f:
            text = f.read()

        documents.append({
            "filename": file,
            "content": text
        })

print(f"Loaded {len(documents)} knowledge base documents.")

Loaded 21 knowledge base documents.


In [5]:
print(documents[0]["filename"])
print("-" * 50)
print(documents[0]["content"][:500])

00_about_this_knowledge_base.md
--------------------------------------------------
---
doc_id: KB-00
title: About this knowledge base
doc_type: meta
audience: agent
language: en
version: 1.2
effective_from: 2026-08-01
status: current
---

# About this knowledge base

## [KB-00.1] Which document wins
When two documents disagree, follow this order of precedence:

1. **Policy documents** (doc_type: policy) with status `current`.
2. **Agent guides** (doc_type: agent_guide).
3. **Help articles** (doc_type: help).
4. **FAQs** (doc_type: faq).
5. **Marketing pages** (doc_type: market


In [6]:
!pip install -q -U \
langchain \
langchain-community \
langchain-text-splitters \
sentence-transformers \
faiss-cpu \
"openai>=2.20,<3.0"

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = []

for doc in documents:
    split_text = text_splitter.split_text(doc["content"])

    for piece in split_text:
        chunks.append({
            "text": piece,
            "source": doc["filename"]
        })

print(f"Created {len(chunks)} text chunks.")

Created 71 text chunks.


In [8]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

print("OpenRouter client initialized successfully!")

OpenRouter client initialized successfully!


In [9]:
#create embeddings 
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(texts)

print(f"Generated {len(embeddings)} embeddings.")
print(f"Embedding dimension: {len(embeddings[0])}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generated 71 embeddings.
Embedding dimension: 384


In [10]:
#create a FIASS database
import faiss
import numpy as np

embedding_array = np.array(embeddings).astype("float32")

index = faiss.IndexFlatL2(embedding_array.shape[1])

index.add(embedding_array)

print(f"FAISS index created with {index.ntotal} vectors.")

FAISS index created with 71 vectors.


In [11]:
query = "Can I return a product after 7 days?"

query_embedding = embedding_model.encode([query])

distances, indices = index.search(
    np.array(query_embedding).astype("float32"),
    k=3
)

print("Top matching chunk indices:")
print(indices)

Top matching chunk indices:
[[38 63 51]]


In [12]:
for i in indices[0]:
    print("=" * 80)
    print(f"Source: {chunks[i]['source']}")
    print()
    print(chunks[i]["text"])
    print()

Source: pol_damaged_wrong_missing.md

## [DWM-4] Reported after 48 hours
Reports after 48 hours are handled as a normal return if the item is still within its return window. Otherwise the customer can use the brand or ZentroMart Assured warranty, if any.

## [DWM-5] Safety issues
If a product shows a safety risk, such as a swelling battery, burning smell, sparks or heat, the customer must stop using it immediately. The agent must hand over to a human at once.

Source: pol_returns_v4.md

## [RET-3] Change to electronics returns
From **1 August 2026**, the return window for electronics and accessories is 7 days. Electronics **delivered before 1 August 2026** keep the earlier 10-day window from version 3 of this policy.

Source: pol_refunds.md

# Refunds policy

## [REF-1] When a refund is initiated
- **Cancelled before shipping:** immediately on cancellation.
- **Returned item:** after the item passes quality check, within 2 business days of pickup. ZentroMart Plus members: at pickup.
- 

In [13]:
def ask_question(question):

    # Create embedding for the user's question
    query_embedding = embedding_model.encode([question])

    # Search FAISS
    distances, indices = index.search(
        np.array(query_embedding).astype("float32"),
        k=3
    )

    # Build context
    context = ""
    sources = []

    for i in indices[0]:
        context += chunks[i]["text"] + "\n\n"
        sources.append(chunks[i]["source"])

    # Prompt
    prompt = f"""
You are ZentroMart's AI customer support assistant.

Answer politely.

Use ONLY the information below.

If the answer isn't available, say:
"I don't know based on the company documents."

Company Documents:
{context}

Customer Question:
{question}
"""

    # Call OpenRouter
    response = client.chat.completions.create(
        model="google/gemini-4-argon",
        max_tokens=300,
        temperature=0.2,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    answer = response.choices[0].message.content

    return answer, sources

In [14]:
answer, sources = ask_question("How long does a refund take?")

print("Answer:")
print(answer)

print("\nSources:")
for source in sources:
    print("-", source)

Answer:
Hello! Thank you for reaching out to ZentroMart customer support.

Refunds usually take 5 to 7 days to reach your account. 

However, the time it takes to initiate the refund depends on the situation:
* **Cancelled before shipping:** The refund is initiated immediately upon cancellation.
* **Returned item:** The refund is initiated within 2 business days of pickup once the item passes the quality check (QC) at our warehouse. If you are a ZentroMart Plus member, the refund is initiated at the time of pickup.
* **Prepaid order refused at the door or not delivered after 3 attempts:** The refund is initiated once the parcel reaches the warehouse, which is usually 7 to 10 calendar days after the last delivery attempt.

Please let me know if I can help you with anything else!

Sources:
- faq_general.md
- pol_returns_v4.md
- pol_refunds.md


In [15]:
tool_path="/kaggle/input/datasets/bolluhansika/capstone-dataset/Capstone_Project_Dataset_FTTT_2026/tools/tools.py"

with open(tool_path,"r") as f:
    print(f.read())

"""Synthetic, offline customer-support tools for the ZentroMart capstone.

The mock order tool gates individual order details behind a simple identity check.
This is an educational simulation, NOT production-ready authentication.
"""
import csv
import os
from datetime import datetime
from uuid import uuid4

_HERE = os.path.dirname(os.path.abspath(__file__))
_DATA = os.path.normpath(os.path.join(_HERE, '..', 'data'))
_TICKETS = os.path.join(_HERE, 'tickets_created.csv')
_TEXT_COLS = {
    'order_id', 'customer_id', 'delivery_pincode', 'return_rrn',
    'cancellation_rrn', 'rto_rrn', 'sku', 'return_sku', 'damage_sku',
    'pincode', 'phone',
}


def _clean(row):
    out = {}
    for key, value in row.items():
        if value == '':
            out[key] = None
        elif value in ('True', 'False'):
            out[key] = value == 'True'
        elif key not in _TEXT_COLS and value.lstrip('-').isdigit():
            out[key] = int(value)
        else:
            out[key] = value
    re

In [16]:
practice_path="/kaggle/input/datasets/bolluhansika/capstone-dataset/Capstone_Project_Dataset_FTTT_2026/practice/questions.csv"

import pandas as pd

practice=pd.read_csv(practice_path)

practice.head(10)

,id,language,question,asr_text,intent,expected_decision,acceptable_decisions,answerable,gold_evidence,hard_negatives,must_include,must_not_include,reference_answer,reply_language,entities,notes,needs_native_review
0,PQ0002,hi-en,SKU ELE-0012 pe ZentroMart Assured hai kya? Cl...,so sku ele-zero zero one to pe zentromart assu...,warranty,answer,answer,True,WAR-2 ; products.csv:ELE-0012,NaN,12 ; My Orders / app,NaN,"Haan, 12 mahine ki Assured warranty hai; claim...",hi-en,sku=ELE-0012,NaN,False
1,PQ0003,en,How fast are refunds to the ZentroMart Wallet?,how fast are refunds to the zentromart wallet,refund_timeline,answer,answer,True,REF-2 ; KB-00.1,FAQ-1,instant / 2 hours / two hours / तुरंत / turant,5 to 7 days ; 5-7 days,"Instant, within 2 hours of initiation.",en,NaN,Trap: FAQ-1 says 5 to 7 days for everything; p...,False
2,PQ0004,en,Can I schedule a delivery at midnight?,uh can i schedule a delivery at midnight,unsupported,answer,answer ; escalate,False,NaN,NaN,don't have / do not have / not sure / no infor...,NaN,"Sorry, I don't have information about that. I ...",en,NaN,Not covered by any document. Do not invent a p...,False
3,PQ0005,hi-en,SKU ELE-0002 pe ZentroMart Assured hai kya? Cl...,uh sku ele-zero zero zero to pe zentromart ass...,warranty,answer,answer,True,WAR-2 ; products.csv:ELE-0002,NaN,12 ; My Orders / app,NaN,"Haan, 12 mahine ki Assured warranty hai; claim...",hi-en,sku=ELE-0002,NaN,False
4,PQ0006,en,"As a ZentroMart Plus member, will I be charged...",uh as a zentromart plus member will i be charg...,return_fee,answer,answer,True,RET-6 ; PLS-2,NaN,no / not / waived / free,₹49 will be charged,No. ZentroMart Plus members pay no reverse pic...,en,NaN,NaN,False
5,PQ0008,hi-en,credit card wala refund 17 October 2026 ko shu...,so credit card wala refund 17 october two zero...,refund_timeline,answer,answer,True,REF-2 ; KB-00.3,NaN,2026-10-27 / 27 October 2026 / 27 अक्टूबर 2026,NaN,24 October 2026 se 27 October 2026 ke beech; S...,hi-en,NaN,card refund: 5 to 7 business days; needs the h...,False
6,PQ0009,en,Is there a student discount?,uh is there a student discount,unsupported,answer,answer ; escalate,False,NaN,NaN,don't have / do not have / not sure / no infor...,NaN,"Sorry, I don't have information about that. I ...",en,NaN,Not covered by any document. Do not invent a p...,False
7,PQ0010,en,Is the collection free if the UrbanThread Slim...,is the free if the urbanthread slim fit jeans ...,return_fee,answer,answer,True,RET-6 ; DWM-2,NaN,free / no fee / waived / no charge,NaN,Yes. No fee applies for damaged or wrong items...,en,NaN,NaN,False
8,PQ0013,hi,"मैंने Gulmohar Matte Lipstick खोल लिया है, क्य...",मैंने Gulmohar Matte Lipstick खोल लिया है क्या...,return_eligibility,answer,answer,True,RET-4,NaN,सील / seal / बंद / खुला,NaN,"नहीं, ब्यूटी प्रोडक्ट सिर्फ़ बिना खोले, सील के...",hi,NaN,NaN,False
9,PQ0014,hi,मुझे Pagecraft A5 Ruled Notebook (Pack of 4) ल...,मुझे Pagecraft A5 Ruled Notebook Pack of 4 लौट...,return_window,answer,answer,True,RET-2 ; products.csv:BKS-0056,NaN,10,NaN,डिलीवरी के 10 दिन के अंदर लौटा सकते हैं।,hi,sku=BKS-0056,NaN,False


In [17]:
orders = pd.read_csv("/kaggle/input/datasets/bolluhansika/capstone-dataset/Capstone_Project_Dataset_FTTT_2026/data/orders.csv")

orders.head()

,order_id,customer_id,placed_at,status,payment_method,delivery_type,delivery_pincode,promised_date,shipped_on,delivered_on,...,return_refund_completed_on,return_rrn,return_reverse_pickup_fee_inr,damage_sku,damage_issue,damage_reported_at,damage_hours_after_delivery,damage_photos_attached,damage_status,timeline
0,69520867,C100124,2026-10-12T08:37,shipped,upi,standard,160017,2026-10-23,2026-10-13,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-10-12T08:37 placed ; 2026-10-12T10:08 pac...
1,54239550,C100102,2026-10-17T07:34,cancelled,upi,standard,641004,2026-10-29,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-10-17T07:34 placed ; 2026-10-17T19:59 can...
2,59591825,C100076,2026-06-11T19:18,delivered,netbanking,slot,411045,2026-06-11,2026-06-11,2026-06-11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-11T19:18 placed ; 2026-06-11T20:16 pac...
3,46637226,C100281,2026-09-30T18:11,delivered,cod,standard,122002,2026-10-08,2026-10-01,2026-10-12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-09-30T18:11 placed ; 2026-09-30T19:23 pac...
4,23048475,C100120,2026-09-02T15:26,delivered,cod,standard,600042,2026-09-09,2026-09-03,2026-09-09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-09-02T15:26 placed ; 2026-09-02T17:09 pac...


In [18]:
customers = pd.read_csv("/kaggle/input/datasets/bolluhansika/capstone-dataset/Capstone_Project_Dataset_FTTT_2026/data/customers.csv")

customers.head()

,customer_id,name,phone,email,city,state,pincode,is_plus,plus_since,preferred_language,created_at
0,C100000,Shruti Das,+91 55846 45060,shruti.das289@example.com,Kolkata,West Bengal,700091,False,NaN,kn,2023-09-02
1,C100001,Abhishek Deshpande,+91 55908 53801,abhishek.deshpande167@example.com,Surat,Gujarat,395007,False,NaN,hi-en,2025-05-03
2,C100002,Rohan Shah,+91 59406 92566,rohan.shah387@example.com,New Delhi,Delhi,110092,True,2025-05-04,ml,2024-12-26
3,C100003,Aadhya Bose,+91 58864 23004,aadhya.bose555@example.com,Dehradun,Uttarakhand,248001,True,2023-11-07,hi,2023-02-28
4,C100004,Suresh Singh,+91 52808 45850,suresh.singh674@example.com,Bengaluru,Karnataka,560076,False,NaN,en,2023-07-03


In [19]:
import sys

sys.path.append("/kaggle/input/datasets/bolluhansika/capstone-dataset/Capstone_Project_Dataset_FTTT_2026")

In [20]:
from tools.tools import lookup_order, create_ticket

In [21]:
print(lookup_order("69520867"))

{'verification_required': True, 'order_id': '69520867', 'message': 'Ask for the registered phone last 4 digits OR delivery pincode.'}


In [22]:
print(lookup_order("69520867", "160017"))

{'found': True, 'verified': True, 'order_id': '69520867', 'customer_id': 'C100124', 'placed_at': '2026-10-12T08:37', 'status': 'shipped', 'payment_method': 'upi', 'delivery_type': 'standard', 'delivery_pincode': '160017', 'promised_date': '2026-10-23', 'shipped_on': '2026-10-13', 'delivered_on': None, 'delivery_attempts': 0, 'subtotal_inr': 3778, 'coupon_code': None, 'coupon_discount_inr': None, 'coupon_min_order_inr': None, 'delivery_fee_inr': 0, 'cod_fee_inr': 0, 'total_inr': 3778, 'contacts_last_7_days': 1, 'cancellation_refund_initiated_on': None, 'cancellation_refund_method': None, 'cancellation_refund_amount_inr': None, 'cancellation_rrn': None, 'rto_refund_initiated_on': None, 'rto_refund_method': None, 'rto_refund_amount_inr': None, 'rto_rrn': None, 'express_late_credit_inr': None, 'express_late_credit_on': None, 'return_sku': None, 'return_line_no': None, 'return_qty': None, 'return_reason': None, 'return_status': None, 'return_requested_on': None, 'return_pickup_on': None, 'r

In [23]:
def decide_action(user_query):

    query = user_query.lower()

    order_keywords = [
        "track", "order", "shipment", "where is my order",
        "delivery status", "my order", "order status"
    ]

    ticket_keywords = [
        "battery", "swelling", "smoke", "fire",
        "sparks", "burning", "danger",
        "human agent", "escalate", "unsafe"
    ]

    for word in ticket_keywords:
        if word in query:
            return "CREATE_TICKET"

    for word in order_keywords:
        if word in query:
            return "ORDER_LOOKUP"

    return "RAG"

In [24]:
print(decide_action("Track my order"))

ORDER_LOOKUP


In [25]:
print(decide_action("My phone battery is swelling"))

CREATE_TICKET


In [26]:
pending_order_id = None


def chat(user_message):
    global pending_order_id

    # -----------------------------
    # Waiting for verification
    # -----------------------------
    if pending_order_id is not None:

        result = lookup_order(pending_order_id, user_message)

        if result.get("verified"):

            print("\nOrder Details")
            print("-----------------------------")
            print("Order ID:", result["order_id"])
            print("Customer:", result["customer_first_name"])
            print("Status:", result["status"])
            print("Payment Method:", result["payment_method"])
            print("Delivery Type:", result["delivery_type"])
            print("Placed At:", result["placed_at"])
            print("Shipped On:", result["shipped_on"])
            print("Delivered On:", result["delivered_on"])
            print("Total: ₹", result["total_inr"])

            print("\nItems:")
            for item in result["items"]:
                print(f"- {item['title']} (Qty: {item['qty']})")

        else:
            print(result["message"])

        # Always reset after verification attempt
        pending_order_id = None
        return

    # -----------------------------
    # Decide intent
    # -----------------------------
    action = decide_action(user_message)

    print("Action:", action)

    # -----------------------------
    # RAG
    # -----------------------------
    if action == "RAG":

        print("Using RAG...\n")

        answer, sources = ask_question(user_message)

        print(answer)

        print("\nSources:")
        for s in sources:
            print("-", s)

    # -----------------------------
    # ORDER LOOKUP
    # -----------------------------
    elif action == "ORDER_LOOKUP":

        import re

        match = re.search(r"\d{8}", user_message)

        if match:

            order_id = match.group()

            result = lookup_order(order_id)

            if result.get("verification_required"):

                pending_order_id = order_id
                print(result["message"])

            elif result.get("verified"):

                print("\nOrder Details")
                print("-----------------------------")
                print("Order ID:", result["order_id"])
                print("Customer:", result["customer_first_name"])
                print("Status:", result["status"])
                print("Payment Method:", result["payment_method"])
                print("Delivery Type:", result["delivery_type"])
                print("Placed At:", result["placed_at"])
                print("Shipped On:", result["shipped_on"])
                print("Delivered On:", result["delivered_on"])
                print("Total: ₹", result["total_inr"])

                print("\nItems:")
                for item in result["items"]:
                    print(f"- {item['title']} (Qty: {item['qty']})")

            else:
                print("Order not found.")

        else:
            print("Please enter your Order ID.")

    # -----------------------------
    # CREATE TICKET
    # -----------------------------
    elif action == "CREATE_TICKET":

        print("This issue requires human support.")

In [27]:
chat("Can I return my laptop after 8 days?")

Action: RAG
Using RAG...

Hello! Thank you for reaching out to ZentroMart customer support.

Whether you can return your laptop after 8 days depends on when it was delivered:

* **Delivered before 1 August 2026:** Yes, you can return it, as electronics delivered before this date have a 10-day return window.
* **From 1 August 2026:** Unfortunately, you would not be able to make a standard return after 8 days, as the return window for electronics is 7 days. (If you are reporting an issue after the return window has closed, you can use the brand or ZentroMart Assured warranty, if any). 

If your laptop is eligible for a return based on the delivery date, please ensure that it meets our return conditions: it must be unused, not physically damaged, have any screen lock or account lock removed, and include all original tags, packaging, accessories, and free gifts. 

Please let me know if I can help you with anything else!

Sources:
- pol_returns_v4.md
- pol_damaged_wrong_missing.md
- pol_ret

In [28]:
chat("What if my package is damaged?")

Action: RAG
Using RAG...

Hello! Thank you for reaching out to ZentroMart customer support.

If an item in your package arrives damaged, please report it within **48 hours of delivery**, along with photos of the item and the packaging. This policy applies to every category, including grocery and other non-returnable items. 

If you report a damaged item after 48 hours, it will be handled as a normal return as long as the item is still within its return window. Otherwise, you can use the brand or ZentroMart Assured warranty, if any applies. 

Additionally, if the product shows a safety risk—such as a swelling battery, burning smell, sparks, or heat—please stop using it immediately, and I will hand this over to a human agent at once.

*(Note: If only the external packaging is damaged and not the item itself, I don't know based on the company documents how that specific situation is handled.)* 

Please let me know if I can assist you further!

Sources:
- pol_damaged_wrong_missing.md
- pol

In [29]:
chat("How long does a refund take?")

Action: RAG
Using RAG...

Hello! Thank you for reaching out to ZentroMart support. 

Refunds usually take 5 to 7 days to reach your account. 

Additionally, please note when a refund is initiated depending on your situation:
* **Cancelled before shipping:** The refund is initiated immediately upon cancellation.
* **Returned item:** The refund is initiated within 2 business days of pickup once the item passes the quality check at the warehouse. (For ZentroMart Plus members, it is initiated at the time of pickup).
* **Prepaid order refused at the door or not delivered after 3 attempts:** The refund is initiated once the parcel reaches the warehouse, which is usually 7 to 10 calendar days after the last delivery attempt.

Please let me know if I can help you with anything else!

Sources:
- faq_general.md
- pol_returns_v4.md
- pol_refunds.md


In [30]:
chat("Who pays the reverse pickup fee?")

Action: RAG
Using RAG...

Hello! Thank you for reaching out to ZentroMart support. 

Based on our company documents, there are two different policies listed regarding the reverse pickup fee:

* **Document [RET-6]:** States that a reverse pickup fee of ₹49 applies to change-of-mind returns for clothing, footwear, and bags. However, this fee is waived for ZentroMart Plus members and for any items returned because they were damaged, defective, or wrong. 
* **Document [RETv3-6]:** States that reverse pickup is free for all returns.

Please let me know if I can help you with anything else!

Sources:
- pol_returns_v4.md
- pol_returns_v3_superseded.md
- help_reverse_pickup.md


In [31]:
chat("Where is my order 69520867?")

Action: ORDER_LOOKUP
Ask for the registered phone last 4 digits OR delivery pincode.


In [32]:
chat("160017")


Order Details
-----------------------------
Order ID: 69520867
Customer: Naveen
Status: shipped
Payment Method: upi
Delivery Type: standard
Placed At: 2026-10-12T08:37
Shipped On: 2026-10-13
Delivered On: None
Total: ₹ 3778

Items:
- Mixwell 750W Mixer Grinder, Green (Qty: 1)
- Tavaa Non-stick Frying Pan 26 cm, Black (Qty: 1)


In [33]:
chat("Cancel my order 69520867")

Action: ORDER_LOOKUP
Ask for the registered phone last 4 digits OR delivery pincode.


In [34]:
chat("Who is the CEO of Google?")

Verification failed or order not found.
